In [2]:
import pandas as pd
import os
curr_df = pd.read_csv('./../data/image_analysis_data/df_with_clutter_features.csv')

In [3]:
curr_df.columns

Index(['keyword', 'source_file', 'asin', 'item_name', 'brand', 'image_count',
       'main_image_url', 'has_aplus', 'has_brand_story', 'review_count',
       'avg_rating', 'bsr_best', 'bsr_paths', 'units_per_month',
       'sales_velocity_daily', 'product_url', 'image_list', 'image_path',
       'edge_density', 'bg_white_pct', 'bg_neutral_pct', 'n_clusters_sig',
       'color_entropy', 'largest_cluster_pct', 'edge_density_z',
       'n_clusters_sig_z', 'color_entropy_z', 'bg_white_pct_z',
       'bg_neutral_pct_z', 'largest_cluster_pct_z', 'clutter_score'],
      dtype='object')

In [4]:
curr_df = curr_df.drop_duplicates()

In [5]:
curr_df['asin']

0        B0BQPNMXQV
1        B0CTBCDD6D
3        B08WM3LMJF
5        B0DGHMNQ5Z
6        B0BS1QCFHX
            ...    
19042    B0F2QVR6TH
19045    B0DF85L4S8
19046    B0FMC49YWD
19047    B0DDYBNHV5
19048    B0D1LNH1DT
Name: asin, Length: 17375, dtype: object

In [6]:
SCRAPINGDOG_API_KEY = os.environ.get('SCRAPINGDOG_API_KEY')

In [7]:
print(SCRAPINGDOG_API_KEY)

68d857ff15822dae285a0810


In [8]:
import os, time, json, re, requests, threading
import pandas as pd
from typing import Any, Dict, List, Optional
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

SCRAPINGDOG_URL = "https://api.scrapingdog.com/amazon/product"
_NUMERIC = re.compile(r"[-+]?\d*[\.,]?\d+")

# -------------------- helpers --------------------
def _get_api_key() -> str:
    k = os.getenv("SCRAPINGDOG_API_KEY")
    if not k:
        raise RuntimeError("SCRAPINGDOG_API_KEY is not set.")
    return k

def _to_float_price(x: Optional[str]) -> Optional[float]:
    if not isinstance(x, str) or not x:
        return None
    s = x.replace(",", "")
    m = _NUMERIC.search(s)
    return float(m.group(0)) if m else None

def _parse_int_like(x: Optional[str]) -> Optional[int]:
    if x is None:
        return None
    s = str(x).replace(",", "")
    m = _NUMERIC.search(s)
    if not m: return None
    try:
        return int(float(m.group(0)))
    except Exception:
        return None

def _parse_stars(x: Optional[str]) -> Optional[float]:
    if x is None:
        return None
    m = _NUMERIC.search(str(x))
    return float(m.group(0)) if m else None

def _parse_k_or_m(x: Optional[str]) -> Optional[int]:
    # '10K+' -> 10000; '1.2M' -> 1_200_000; '532' -> 532
    if not isinstance(x, str) or not x:
        return None
    s = x.upper().replace(",", "")
    m = _NUMERIC.search(s)
    if not m:
        return None
    val = float(m.group(0))
    if "M" in s:
        val *= 1_000_000
    elif "K" in s:
        val *= 1_000
    return int(round(val))

def _drop_heavy_fields(d: Dict[str, Any]) -> Dict[str, Any]:
    skip = {
        "description", "main_image", "images", "images_of_specified_asin",
        "customers_freuqently_viewed", "products_related_to_this_item",
        "customer_who_bought_this_item_also_bought", "customization_options",
        "author", "feature_bullets"
    }
    return {k: v for k, v in d.items() if k not in skip}

# ---- JSON/shape coercers + error logging ----
def _maybe_json(obj):
    if isinstance(obj, str):
        s = obj.strip()
        if s and (s.startswith("{") or s.startswith("[") or s.startswith('"')):
            try:
                return json.loads(s)
            except Exception:
                return obj
    return obj

def _as_dict(obj, asin: str, field: str, errors: list) -> dict:
    obj = _maybe_json(obj)
    if isinstance(obj, dict):
        return obj
    if isinstance(obj, str):
        # string that wasn't valid JSON dict
        errors.append({"asin": asin, "type": "schema", "field": field, "error": "expected dict, got str", "value": obj[:200]})
        return {}
    if obj not in (None, {}):
        errors.append({"asin": asin, "type": "schema", "field": field, "error": f"expected dict, got {type(obj).__name__}", "value": str(obj)[:200]})
    return {}

def _as_list(obj, asin: str, field: str, errors: list) -> list:
    obj = _maybe_json(obj)
    if isinstance(obj, list):
        return obj
    if isinstance(obj, str):
        errors.append({"asin": asin, "type": "schema", "field": field, "error": "expected list, got str", "value": obj[:200]})
        return []
    if obj not in (None, []):
        errors.append({"asin": asin, "type": "schema", "field": field, "error": f"expected list, got {type(obj).__name__}", "value": str(obj)[:200]})
    return []

# ---------------- thread-local session ----------------
_TLS = threading.local()
def _get_session() -> requests.Session:
    s = getattr(_TLS, "sess", None)
    if s is None:
        s = requests.Session()
        adapter = requests.adapters.HTTPAdapter(pool_connections=100, pool_maxsize=100, max_retries=0)
        s.mount("https://", adapter)
        s.mount("http://", adapter)
        _TLS.sess = s
    return s

def _scrape_one(api_key: str, asin: str, retry: int = 1, timeout: int = 30) -> Dict[str, Any]:
    params = {"api_key": api_key, "asin": asin, "domain": "com", "postal_code": "", "country": "us"}
    last_err = None
    for attempt in range(retry + 1):
        try:
            sess = _get_session()
            r = sess.get(SCRAPINGDOG_URL, params=params, timeout=timeout)
            if r.status_code == 200:
                return r.json()
            last_err = RuntimeError(f"ScrapingDog {asin} -> {r.status_code}: {r.text[:200]}")
        except Exception as e:
            last_err = e
        if attempt < retry:
            time.sleep(1.5 * (attempt + 1))
    return {"_error": str(last_err) if last_err else "Unknown error"}

# ---------------- parallel enrich (returns df AND errors) ----------------
def enrich_common_fields_parallel(
    curr_df: pd.DataFrame,
    n: int = 100,
    max_workers: int = 8,
    retry: int = 1,
):
    """
    Parallel enrichment with robust schema handling.
    Returns: merged_df, errors_df
    - merged_df includes slim sd_* columns and a per-row 'sd_errors' (list) if any.
    - errors_df lists all issues (network + schema) with asin/field/type/error/value.
    """
    if "asin" not in curr_df.columns:
        raise ValueError("curr_df must contain an 'asin' column.")

    api_key = _get_api_key()
    asins = curr_df["asin"].astype(str).dropna().drop_duplicates().head(n).tolist()

    rows: List[Dict[str, Any]] = []
    all_errors: List[Dict[str, Any]] = []

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        fut_to_asin = {ex.submit(_scrape_one, api_key, a, retry): a for a in asins}
        for fut in tqdm(as_completed(fut_to_asin), total=len(fut_to_asin), desc="Enriching ASINs", unit="asin"):
            a = fut_to_asin[fut]
            data = fut.result()
            row = {"asin": a}
            row_errors = []

            # Network/API error captured
            if "_error" in (data or {}):
                err = {"asin": a, "type": "network", "field": "request", "error": data["_error"], "value": ""}
                all_errors.append(err)
                row["sd_error"] = data["_error"]
                row["sd_errors"] = [err["error"]]
                rows.append(row)
                continue

            try:
                d = _drop_heavy_fields(data or {})

                # Basic common fields
                row["sd_title"] = d.get("title")
                row["sd_parent_asin"] = d.get("parent_asin")
                row["sd_price"] = _to_float_price(d.get("price"))
                row["sd_list_price"] = _to_float_price(d.get("list_price"))
                row["sd_previous_price"] = _to_float_price(d.get("previous_price"))
                row["sd_price_symbol"] = d.get("price_symbol")
                row["sd_availability_status"] = d.get("availability_status")
                row["sd_aplus"] = bool(d.get("aplus"))
                row["sd_is_prime_exclusive"] = bool(d.get("is_prime_exclusive"))
                row["sd_is_frequently_returned"] = bool(d.get("is_frequently_returned"))
                row["sd_number_bought_past_month"] = _parse_k_or_m(d.get("number_of_people_bought"))

                row["sd_average_rating"] = _parse_stars(d.get("average_rating"))
                row["sd_total_reviews"] = _parse_int_like(d.get("total_reviews"))

                # product_information & nested Customer Reviews → can be strings or dicts
                pi = _as_dict(d.get("product_information"), a, "product_information", all_errors)
                cr = _as_dict(pi.get("Customer Reviews"), a, "Customer Reviews", all_errors)

                row["sd_ratings_count"] = _parse_int_like(cr.get("ratings_count"))
                row["sd_stars"] = _parse_stars(cr.get("stars"))

                # ratings_distribution & customer_sentiments → string or list
                rd = _as_list(d.get("ratings_distribution"), a, "ratings_distribution", all_errors)
                cs = _as_list(d.get("customer_sentiments"), a, "customer_sentiments", all_errors)

                row["sd_ratings_distribution"] = json.dumps(rd, ensure_ascii=False)
                row["sd_customer_sentiments"] = json.dumps(cs, ensure_ascii=False)

            except Exception as e:
                err = {"asin": a, "type": "schema", "field": "parse_row", "error": f"{type(e).__name__}: {e}", "value": ""}
                all_errors.append(err)
                row_errors.append(err["error"])

            # attach compact per-row errors list (if any)
            if row_errors:
                row["sd_errors"] = row_errors

            rows.append(row)

    enrich_df = pd.DataFrame(rows)

    # Build errors_df
    errors_df = pd.DataFrame(all_errors) if all_errors else pd.DataFrame(columns=["asin","type","field","error","value"])

    # Prefer successful rows; avoid MergeError by deduping
    if not enrich_df.empty:
        ok_mask = ~enrich_df.get("sd_error", pd.Series([False]*len(enrich_df))).notna()
        enrich_df["_ok"] = ok_mask
        enrich_df["_rev"] = enrich_df.get("sd_total_reviews")
        enrich_df = (
            enrich_df
            .sort_values(["_ok", "_rev"], ascending=[False, False])
            .drop(columns=["_ok", "_rev"], errors="ignore")
            .drop_duplicates(subset="asin", keep="first")
        )

    merged_df = curr_df.merge(enrich_df, on="asin", how="left", validate="m:1")
    return merged_df, errors_df


In [11]:
enriched_df, errors_df = enrich_common_fields_parallel(curr_df, n=len(curr_df), max_workers=5, retry=1)

[x for x in enriched_df.columns if x.startswith("sd_")][:20]
enriched_df.head(2)


Enriching ASINs: 100%|██████████| 17295/17295 [3:07:32<00:00,  1.54asin/s]  


,keyword,source_file,asin,item_name,brand,image_count,main_image_url,has_aplus,has_brand_story,review_count,...,sd_is_frequently_returned,sd_number_bought_past_month,sd_average_rating,sd_total_reviews,sd_ratings_count,sd_stars,sd_ratings_distribution,sd_customer_sentiments,sd_error,sd_errors
0,audio headphones catalog full 1757641194,audio_headphones_catalog_full_1757641194.csv,B0BQPNMXQV,JBL Vibe Beam - True Wireless JBL Deep Bass So...,JBL,18,https://m.media-amazon.com/images/I/31S4tOQj4S...,False,False,NaN,...,False,10000000.0,NaN,NaN,NaN,NaN,[],[],NaN,NaN
1,audio headphones catalog full 1757641194,audio_headphones_catalog_full_1757641194.csv,B0CTBCDD6D,JBL Tune 720BT - Wireless Over-Ear Headphones ...,JBL,21,https://m.media-amazon.com/images/I/61EL2AKKcB...,False,False,NaN,...,False,8000000.0,4.5,7854.0,7854.0,4.5,"[{""rating"": 5, ""distribution"": ""76""}, {""rating...","[{""title"": ""Sound quality"", ""sentiment"": ""POSI...",NaN,NaN


In [16]:
enriched_df.loc[enriched_df["sd_errors"].notna(), ["asin", "sd_errors"]].head()


,asin,sd_errors
63,B083W7V3RG,"[ScrapingDog B083W7V3RG -> 404: {""message"":""Pr..."
364,B0FLVCVLWN,"[ScrapingDog B0FLVCVLWN -> 404: {""message"":""Pr..."
3632,B01MXFA41N,"[ScrapingDog B01MXFA41N -> 400: {""message"":""So..."
3717,B094PS5RZQ,"[ScrapingDog B094PS5RZQ -> 400: {""message"":""So..."
4072,B0DSM47DCW,"[ScrapingDog B0DSM47DCW -> 400: {""message"":""So..."


In [27]:
enriched_df.to_csv('./../data/data_with_scraper.csv')

In [28]:
import json
import re
import pandas as pd
from typing import Any, Dict, Iterable, List, Optional

def _parse_json_cell(cell):
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return None
    if isinstance(cell, (list, dict)):
        return cell
    if isinstance(cell, str) and cell.strip():
        try:
            return json.loads(cell)
        except Exception:
            return None
    return None

def _to_float(x) -> Optional[float]:
    try:
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return None
        return float(str(x).replace("%", "").replace(",", "").strip())
    except Exception:
        return None

def unpack_ratings_distribution(df: pd.DataFrame,
                                src_col: str = "sd_ratings_distribution",
                                prefix: str = "sd_rating_pct_") -> pd.DataFrame:
    target_cols = [f"{prefix}{i}" for i in range(1, 6)]
    for c in target_cols:
        if c not in df.columns:
            df[c] = pd.NA

    # Fill row-by-row
    for i, cell in df[src_col].items() if src_col in df.columns else []:
        parsed = _parse_json_cell(cell)
        if not parsed:
            continue
        m = {}
        for entry in parsed:
            try:
                r = int(entry.get("rating"))
            except Exception:
                continue
            pct = _to_float(entry.get("distribution"))
            if pct is not None and 1 <= r <= 5:
                m[r] = pct
        for r in range(1, 6):
            col = f"{prefix}{r}"
            if r in m:
                df.at[i, col] = m[r]

    for c in target_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    return df

def _slugify_title(title: str) -> str:
    s = re.sub(r"\s+", "_", str(title).strip())
    s = re.sub(r"[^A-Za-z0-9_]", "", s)
    return s or "Unknown"

def unpack_customer_sentiments(df: pd.DataFrame,
                               src_col: str = "sd_customer_sentiments",
                               col_prefix: str = "sd_sent_",
                               add_sent_counts: bool = True) -> pd.DataFrame:
    if src_col not in df.columns:
        return df

    all_titles: List[str] = []
    for cell in df[src_col]:
        parsed = _parse_json_cell(cell)
        if not parsed:
            continue
        for entry in parsed:
            t = entry.get("title")
            if t:
                all_titles.append(str(t))
    unique_titles = sorted(set(all_titles))


    col_map = {}
    for t in unique_titles:
        colname = f"{col_prefix}{_slugify_title(t)}"
        col_map[t] = colname
        if colname not in df.columns:
            df[colname] = pd.NA

    if add_sent_counts:
        for lab in ("POSITIVE", "MIXED", "NEGATIVE"):
            c = f"{col_prefix}count_{lab}"
            if c not in df.columns:
                df[c] = 0


    for i, cell in df[src_col].items():
        parsed = _parse_json_cell(cell)
        if not parsed:
            continue
        # local counts
        pos = mix = neg = 0
        for entry in parsed:
            t = entry.get("title")
            s = entry.get("sentiment")
            if not t:
                continue
            colname = col_map.get(str(t))
            if colname:
                df.at[i, colname] = s
            if add_sent_counts and isinstance(s, str):
                ss = s.upper()
                if ss == "POSITIVE":
                    pos += 1
                elif ss == "MIXED":
                    mix += 1
                elif ss == "NEGATIVE":
                    neg += 1
        if add_sent_counts:
            df.at[i, f"{col_prefix}count_POSITIVE"] = pos
            df.at[i, f"{col_prefix}count_MIXED"] = mix
            df.at[i, f"{col_prefix}count_NEGATIVE"] = neg

    return df




In [52]:
enriched_df = pd.read_csv('./../data/data_with_scraper.csv')

In [54]:
enriched_df.drop(columns='Unnamed: 0', inplace=True)

In [55]:
enriched_df = unpack_ratings_distribution(enriched_df, src_col="sd_ratings_distribution")
#enriched_df = unpack_customer_sentiments(enriched_df, src_col="sd_customer_sentiments")

In [43]:
enriched_df.columns

Index(['keyword', 'source_file', 'asin', 'item_name', 'brand', 'image_count',
       'main_image_url', 'has_aplus', 'has_brand_story', 'review_count',
       'avg_rating', 'bsr_best', 'bsr_paths', 'units_per_month',
       'sales_velocity_daily', 'product_url', 'image_list', 'image_path',
       'edge_density', 'bg_white_pct', 'bg_neutral_pct', 'n_clusters_sig',
       'color_entropy', 'largest_cluster_pct', 'edge_density_z',
       'n_clusters_sig_z', 'color_entropy_z', 'bg_white_pct_z',
       'bg_neutral_pct_z', 'largest_cluster_pct_z', 'clutter_score',
       'sd_title', 'sd_parent_asin', 'sd_price', 'sd_list_price',
       'sd_previous_price', 'sd_price_symbol', 'sd_availability_status',
       'sd_aplus', 'sd_is_prime_exclusive', 'sd_is_frequently_returned',
       'sd_number_bought_past_month', 'sd_average_rating', 'sd_total_reviews',
       'sd_ratings_count', 'sd_stars', 'sd_ratings_distribution',
       'sd_customer_sentiments', 'sd_rating_pct_1', 'sd_rating_pct_2',
       's

In [39]:
enriched_df['sd_rating_pct_5'].tail(20)

17355     0.0
17356     NaN
17357     0.0
17358     NaN
17359     0.0
17360     0.0
17361    65.0
17362     0.0
17363     NaN
17364     NaN
17365     0.0
17366    64.0
17367     0.0
17368     0.0
17369     NaN
17370     NaN
17371     NaN
17372     0.0
17373     0.0
17374     0.0
Name: sd_rating_pct_5, dtype: float64

In [57]:
enriched_df.columns

Index(['keyword', 'source_file', 'asin', 'item_name', 'brand', 'image_count',
       'main_image_url', 'has_aplus', 'has_brand_story', 'review_count',
       'avg_rating', 'bsr_best', 'bsr_paths', 'units_per_month',
       'sales_velocity_daily', 'product_url', 'image_list', 'image_path',
       'edge_density', 'bg_white_pct', 'bg_neutral_pct', 'n_clusters_sig',
       'color_entropy', 'largest_cluster_pct', 'edge_density_z',
       'n_clusters_sig_z', 'color_entropy_z', 'bg_white_pct_z',
       'bg_neutral_pct_z', 'largest_cluster_pct_z', 'clutter_score',
       'sd_title', 'sd_parent_asin', 'sd_price', 'sd_list_price',
       'sd_previous_price', 'sd_price_symbol', 'sd_availability_status',
       'sd_aplus', 'sd_is_prime_exclusive', 'sd_is_frequently_returned',
       'sd_number_bought_past_month', 'sd_average_rating', 'sd_total_reviews',
       'sd_ratings_count', 'sd_stars', 'sd_ratings_distribution',
       'sd_customer_sentiments', 'sd_error', 'sd_errors', 'sd_rating_pct_1',
   

In [61]:
enriched_df.to_csv('./../data/17k_products_amazon_data.csv')